# Advanced Training and Production Inference

<div style="display: flex; gap: 10px; margin-bottom: 20px;">
  <a href="https://colab.research.google.com/github/OpenTabular/DeepTab/blob/main/docs/tutorials/notebooks/advanced_training.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
  <a href="https://github.com/OpenTabular/DeepTab/blob/main/docs/tutorials/notebooks/advanced_training.ipynb" target="_blank">
    <img src="https://img.shields.io/badge/View%20on-GitHub-181717?logo=github&logoColor=white" alt="View on GitHub"/>
  </a>
</div>

This tutorial covers the parts of DeepTab you reach for once the basics feel
comfortable: tuning the optimizer, controlling the learning-rate schedule,
plugging in your own optimizer or scheduler, and deploying a trained model with
`InferenceModel`. The sections are self-contained, so feel free to jump straight
to the topic you need.

**What You Will Learn**

- How to discover available optimizers and schedulers at runtime.
- How to pass `optimizer_type`, `optimizer_kwargs`, and scheduler fields through `TrainerConfig`.
- What `no_weight_decay_for_bias_and_norm` does and when to use it.
- How to register a custom optimizer or scheduler.
- How to use `InferenceModel` for schema-validated, deployment-friendly inference.

## Setup

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from deeptab import InferenceModel
from deeptab.configs import MambularConfig, PreprocessingConfig, TrainerConfig
from deeptab.models import MambularClassifier
from deeptab.training import (
    available_optimizers,
    available_schedulers,
    register_optimizer,
    register_scheduler,
    unregister_optimizer,
    unregister_scheduler,
)


```{note}
For a quick demonstration these tutorials train with very low `max_epochs` and `patience` (5 and 2). Treat these as placeholders and choose values that match your own compute budget and problem. As a starting point, at least `max_epochs=100` and `patience=10` are recommended for meaningful results.
```

In [2]:
import logging
import warnings

# These tutorials use small synthetic datasets and short training runs, which
# surfaces a few non-actionable framework messages. Quieten them so the output
# stays focused on the tutorial; none of them affect correctness.
warnings.filterwarnings("ignore", message=".*n_quantiles.*")
warnings.filterwarnings("ignore", message=".*does not have many workers.*")
warnings.filterwarnings("ignore", message=".*have no logger configured.*")
warnings.filterwarnings("ignore", message=".*lr_patience.*")
warnings.filterwarnings("ignore", message=".*Checkpoint directory.*")
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)

## Data

In [3]:
RANDOM_STATE = 42

X_num, y = make_classification(
    n_samples=1500,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    random_state=RANDOM_STATE,
)

X = pd.DataFrame(X_num, columns=[f"feat_{i}" for i in range(X_num.shape[1])])

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE
)

---

## Part 1: Optimizers

The optimizer decides how each gradient update turns into a change in the model's
weights. DeepTab defaults to Adam, a dependable starting point for most tabular
problems. When you want more control, you can select any optimizer in the
registry and forward custom arguments to it through `TrainerConfig`.

### Discovering available optimizers

In [4]:
print(available_optimizers())

['adadelta', 'adagrad', 'adam', 'adamax', 'adamw', 'asgd', 'lbfgs', 'nadam', 'radam', 'rmsprop', 'rprop', 'sgd', 'sparseadam']


### Using AdamW with custom kwargs

In [5]:
trainer = TrainerConfig(
    max_epochs=5,
    batch_size=128,
    lr=3e-4,
    patience=2,
    optimizer_type="adamw",
    optimizer_kwargs={
        "betas": (0.9, 0.98),
        "eps": 1e-8,
    },
    weight_decay=1e-2,
)

clf = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=trainer,
    random_state=RANDOM_STATE,
)
clf.fit(X_train, y_train, X_val=X_val, y_val=y_val)
print("AdamW AUROC:", roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1]))

Numerical Feature: feat_0, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_1, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_2, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_3, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_4, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_5, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
-------------------------

### Weight-decay exemption for bias and normalisation layers

`no_weight_decay_for_bias_and_norm=True` splits the model parameters into two groups:
one with `weight_decay` as configured and one (biases and normalisation weights) with
`weight_decay=0`. This is the recommended practice for transformer-style architectures.

In [6]:
clf_wd = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=TrainerConfig(
        max_epochs=5,
        batch_size=128,
        lr=3e-4,
        patience=2,
        optimizer_type="AdamW", # Case-insensitive, should work the same as "adamw"
        weight_decay=1e-2,
        no_weight_decay_for_bias_and_norm=True,
    ),
    random_state=RANDOM_STATE,
)
clf_wd.fit(X_train, y_train, X_val=X_val, y_val=y_val)
print("AdamW + no-WD-BN AUROC:", roc_auc_score(y_test, clf_wd.predict_proba(X_test)[:, 1]))

Numerical Feature: feat_0, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_1, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_2, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_3, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_4, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_5, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
-------------------------

---

## Part 2: Schedulers

A scheduler adjusts the learning rate as training progresses, and a good schedule
often matters as much as the optimizer itself. A higher rate early on lets the
model make rapid progress, while a lower rate later helps it settle into a good
solution instead of bouncing around it.

### Discovering available schedulers

In [7]:
print(available_schedulers())

['constantlr', 'cosineannealinglr', 'cosineannealingwarmrestarts', 'cycliclr', 'exponentiallr', 'linearlr', 'multisteplr', 'onecyclelr', 'reducelronplateau', 'sequentiallr', 'steplr']


### CosineAnnealingLR

Cosine annealing lowers the learning rate from its starting value toward
`eta_min` along a cosine curve spread over `T_max` epochs. It needs very little
tuning and is a strong default when you train for a fixed number of epochs.

In [8]:
clf_cos = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=TrainerConfig(
        max_epochs=5,
        batch_size=128,
        lr=3e-4,
        patience=2,
        optimizer_type="AdamW",
        weight_decay=1e-2,
        scheduler_type="CosineAnnealingLR",
        scheduler_kwargs={"T_max": 5, "eta_min": 1e-6},
        scheduler_interval="epoch",
    ),
    random_state=RANDOM_STATE,
)
clf_cos.fit(X_train, y_train, X_val=X_val, y_val=y_val)
print("CosineAnnealingLR AUROC:", roc_auc_score(y_test, clf_cos.predict_proba(X_test)[:, 1]))

Numerical Feature: feat_0, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_1, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_2, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_3, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_4, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_5, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
-------------------------

### ReduceLROnPlateau (default)

In [10]:
clf_plateau = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=TrainerConfig(
        max_epochs=5,
        batch_size=128,
        lr=3e-4,
        patience=2,
        optimizer_type="AdamW",
        weight_decay=1e-2,
        scheduler_type="ReduceLROnPlateau",
        scheduler_monitor="val_loss",
        scheduler_kwargs={
            "factor": 0.5,
            "patience": 5,
            "min_lr": 1e-6,
        },
    ),
    random_state=RANDOM_STATE,
)
clf_plateau.fit(X_train, y_train, X_val=X_val, y_val=y_val)
print("ReduceLROnPlateau AUROC:", roc_auc_score(y_test, clf_plateau.predict_proba(X_test)[:, 1]))

Numerical Feature: feat_0, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_1, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_2, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_3, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_4, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_5, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
-------------------------

### Disabling the scheduler

In [11]:
clf_const_lr = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=TrainerConfig(
        max_epochs=5,
        batch_size=128,
        lr=3e-4,
        patience=2,
        scheduler_type=None,
    ),
    random_state=RANDOM_STATE,
)
clf_const_lr.fit(X_train, y_train, X_val=X_val, y_val=y_val)
print("Constant LR AUROC:", roc_auc_score(y_test, clf_const_lr.predict_proba(X_test)[:, 1]))

Numerical Feature: feat_0, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_1, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_2, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_3, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_4, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_5, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
-------------------------

---

## Part 3: Custom Optimizer and Scheduler Registration

Sometimes the built-in choices are not enough, whether you are reproducing a
paper or experimenting with an idea of your own. The registry pattern lets you
plug in any optimizer or scheduler that follows the standard
`torch.optim.Optimizer` or `torch.optim.lr_scheduler.LRScheduler` interface.

### How the registry works

DeepTab keeps a process-global mapping of `name -> class` for optimizers and
another for schedulers. When you pass `optimizer_type="adamw"` to
`TrainerConfig`, DeepTab simply looks that name up in the registry. Three
functions act on it:

- `register_optimizer(name, cls)` / `register_scheduler(name, cls)` — add a new
  entry.
- `available_optimizers()` / `available_schedulers()` — list what is registered.
- `unregister_optimizer(name)` / `unregister_scheduler(name)` — remove an entry
  **you added**.

Two rules keep this safe to use:

- **Names are unique.** Registering a name that already exists raises a
  `ValueError`. Pass `override=True` to intentionally replace it — useful when
  you iterate on an implementation and re-run the cell, or want to swap a
  built-in for your own variant.
- **Built-ins are protected.** You can *override* a built-in like `adam`, but
  you cannot `unregister` it — removing it would break every estimator in the
  process. Only names you registered yourself can be removed.

### Registering a custom optimizer

We pass `override=True` so re-running this cell is safe (otherwise the second
run raises *"Optimizer 'scaledadam' is already registered"*).


In [12]:
class ScaledAdam(torch.optim.Adam):
    """Adam with gradient pre-scaling (toy example)."""

    def __init__(self, params, lr=1e-3, scale=1.0, **kwargs):
        super().__init__(params, lr=lr * scale, **kwargs)


# Names are stored lowercase; lookups are case insensitive.
# override=True makes this cell idempotent: re-running it replaces the
# existing entry instead of raising "already registered".
register_optimizer("scaledadam", ScaledAdam, override=True)
print("scaledadam registered:", "scaledadam" in available_optimizers())

clf_custom_opt = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=TrainerConfig(
        max_epochs=5,
        batch_size=128,
        lr=3e-4,
        patience=2,
        optimizer_type="scaledadam",
        optimizer_kwargs={"scale": 0.8},
    ),
    random_state=RANDOM_STATE,
)
clf_custom_opt.fit(X_train, y_train, X_val=X_val, y_val=y_val)


scaledadam registered: True
Numerical Feature: feat_0, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_1, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_2, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_3, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_4, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_5, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': Non

MambularClassifier(model_config=MambularConfig(head_layer_sizes=[], n_layers=3),
                   preprocessing_config=PreprocessingConfig(numerical_preprocessing='quantile',
                                                            categorical_preprocessing=None,
                                                            n_bins=None,
                                                            feature_preprocessing=None,
                                                            use_decision_tree_bins=None,
                                                            binning_strategy=None,
                                                            task=None,
                                                            cat_cutoff=None,
                                                            treat_all_integers_as_numerical=None,
                                                            degree=None,...
                                                monitor='val_loss',
                                                mode='min',
                                                lr=0.0003,
                                                lr_patience=10,
                                                lr_factor=0.1,
                                                weight_decay=1e-06,
                                                optimizer_type='scaledadam',
                                                optimizer_kwargs={'scale': 0.8},
                                                scheduler_type='ReduceLROnPlateau',
                                                scheduler_kwargs=None,
                                                scheduler_monitor=None,
                                                scheduler_interval='epoch',
                                                scheduler_frequency=1,
                                                no_weight_decay_for_bias_and_norm=False,
                                                checkpoint_path='model_checkpoints'))

### Registering a custom scheduler

Schedulers follow exactly the same rules — `override=True` for idempotent
re-registration, and the same protection for built-ins.


In [13]:
class WarmupConstant(torch.optim.lr_scheduler.LambdaLR):
    """Linear warmup for `warmup_steps`, then constant LR."""

    def __init__(self, optimizer, warmup_steps: int = 100):
        def _lambda(step: int) -> float:
            if step < warmup_steps:
                return float(step) / max(1, warmup_steps)
            return 1.0

        super().__init__(optimizer, lr_lambda=_lambda)


register_scheduler("warmupconstant", WarmupConstant, override=True)
print("warmupconstant registered:", "warmupconstant" in available_schedulers())

clf_warmup = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=TrainerConfig(
        max_epochs=5,
        batch_size=128,
        lr=3e-4,
        patience=2,
        scheduler_type="warmupconstant",
        scheduler_kwargs={"warmup_steps": 200},
        scheduler_interval="step",
    ),
    random_state=RANDOM_STATE,
)
clf_warmup.fit(X_train, y_train, X_val=X_val, y_val=y_val)


warmupconstant registered: True
Numerical Feature: feat_0, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_1, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_2, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_3, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_4, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories': None}
--------------------------------------------------
Numerical Feature: feat_5, Info: {'preprocessing': 'imputer -> minmax -> quantile', 'dimension': 1, 'categories':

MambularClassifier(model_config=MambularConfig(head_layer_sizes=[], n_layers=3),
                   preprocessing_config=PreprocessingConfig(numerical_preprocessing='quantile',
                                                            categorical_preprocessing=None,
                                                            n_bins=None,
                                                            feature_preprocessing=None,
                                                            use_decision_tree_bins=None,
                                                            binning_strategy=None,
                                                            task=None,
                                                            cat_cutoff=None,
                                                            treat_all_integers_as_numerical=None,
                                                            degree=None,...
                                                monitor='val_loss',
                                                mode='min',
                                                lr=0.0003,
                                                lr_patience=10,
                                                lr_factor=0.1,
                                                weight_decay=1e-06,
                                                optimizer_type='Adam',
                                                optimizer_kwargs=None,
                                                scheduler_type='warmupconstant',
                                                scheduler_kwargs={'warmup_steps': 200},
                                                scheduler_monitor=None,
                                                scheduler_interval='step',
                                                scheduler_frequency=1,
                                                no_weight_decay_for_bias_and_norm=False,
                                                checkpoint_path='model_checkpoints'))

### Cleaning up: unregistering your entries

If you no longer need a custom optimizer or scheduler — for example to free up
a name or reset state between experiments — remove it with
`unregister_optimizer` / `unregister_scheduler`. Use `missing_ok=True` for
idempotent teardown that won't raise if the entry is already gone. Built-in
DeepTab names are protected and cannot be removed.


In [14]:
# Remove the custom entries we added above.
unregister_optimizer("scaledadam")
unregister_scheduler("warmupconstant")
print("scaledadam still registered:", "scaledadam" in available_optimizers())

# Safe to call again — missing_ok avoids an error if it is already gone.
unregister_optimizer("scaledadam", missing_ok=True)

# Built-ins are protected: this raises, by design.
try:
    unregister_optimizer("adam")
except ValueError as err:
    print("Refused to remove built-in:", err)


scaledadam still registered: False
Refused to remove built-in: Optimizer 'adam' is a built-in DeepTab optimizer and cannot be unregistered. Built-ins can be replaced with register_optimizer(..., override=True) but not removed.


---

## Part 4: Production Inference with `InferenceModel`

`InferenceModel` wraps a fitted estimator and exposes only the prediction
surface. Training methods (`fit`, `optimize_hparams`, etc.) are absent.

### Save a model to disk

In [15]:
clf_wd.save("advanced_clf.deeptab")

'advanced_clf.deeptab'

### Load via `from_path`

In [16]:
model = InferenceModel.from_path("advanced_clf.deeptab")
print(model)
print("Task:", model.task)
print("Features:", model.n_features)

InferenceModel(task='classification', estimator='MambularClassifier', n_features=12, features=['feat_0', 'feat_1', 'feat_2', ...], n_classes=2)
Task: classification
Features: 12


### Wrap an already-fitted estimator

In [17]:
model_live = InferenceModel.from_estimator(clf_wd)
print(model_live.task)
print(model_live.n_features)

classification
12


### Introspection

In [18]:
info = model.describe()
print(list(info.keys()))

rt = model.runtime_info()
print(list(rt.keys()))

['estimator', 'architecture', 'task', 'built', 'fitted', 'model_config', 'preprocessing_config', 'trainer_config', 'feature_counts', 'num_classes', 'family', 'returns_ensemble', 'parameters', 'inference_task']
['built', 'fitted', 'device', 'dtype', 'precision', 'accelerator', 'strategy', 'num_devices', 'root_device', 'max_epochs', 'current_epoch', 'global_step', 'batch_size', 'optimizer_type', 'lr', 'weight_decay', 'logger', 'deterministic']


### Schema validation

In [19]:
# Happy path
X_clean = model.validate_input(X_test)
print("Schema valid, shape:", X_clean.shape)

# Missing column
X_bad = X_test.drop(columns=["feat_0"])
try:
    model.validate_input(X_bad)
except ValueError as exc:
    print("Missing column error:", exc)

# Extra columns are dropped with a warning in lenient mode
X_wide = X_test.copy()
X_wide["audit_id"] = range(len(X_test))
X_clean = model.validate_input(X_wide, allow_extra_columns=True)
print("After dropping extra column, shape:", X_clean.shape)

Schema valid, shape: (225, 12)
Missing column error: Input is missing 1 column(s) that were present during training: ['feat_0'].
After dropping extra column, shape: (225, 12)


/var/folders/wk/cnjwpb6n7hb63kvw2r5728qh0000gn/T/ipykernel_20074/1621211756.py:15: UserWarning: Input has 1 column(s) not seen during training (['audit_id']); they will be dropped.
  X_clean = model.validate_input(X_wide, allow_extra_columns=True)


### Prediction

In [20]:
# Hard class labels
labels = model.predict(X_clean)
print("Accuracy:", accuracy_score(y_test, labels))

# Class probabilities
proba = model.predict_proba(X_clean)
print("AUROC:", roc_auc_score(y_test, proba[:, 1]))

Accuracy: 0.72
AUROC: 0.7953539823008849


### Production service pattern

A minimal service handler using `InferenceModel`:

In [21]:
# Module-level: load once at startup
_MODEL = InferenceModel.from_path("advanced_clf.deeptab")


def score(payload: dict) -> dict:
    X = pd.DataFrame([payload])
    X_clean = _MODEL.validate_input(X, allow_extra_columns=True)
    proba   = _MODEL.predict_proba(X_clean)
    label   = _MODEL.predict(X_clean)
    return {
        "probability_positive": float(proba[0, 1]),
        "label": int(label[0]),
    }


# Example call
sample = X_test.iloc[0].to_dict()
result = score(sample)
print(result)

{'probability_positive': 0.5087212324142456, 'label': 1}


---

## Next Steps

- [Core concepts: training and evaluation](../../core_concepts/training_and_evaluation)
- [Core concepts: inference](../../core_concepts/inference)
- [Imbalanced classification tutorial](imbalance_classification)
- [Skewed-target regression](skewed_regression)